# 02_baseline_models.ipynb  
## Baseline Modeling using Processed Datasets from Google Drive

### Overview
In this notebook, we are performing baseline modeling as the second stage of our PhD thesis data pipeline. The processed datasets generated during the preprocessing stage have been stored in Google Drive under the directory:

MyDrive/PhD_Thesis_DataPipeline/Processed Datasets Bundles/

Unlike earlier attempts, we are now directly accessing the datasets from Google Drive using a verified and corrected directory structure. This ensures that all dataset folders are correctly detected and eliminates issues related to missing directories or incorrect paths.

### Objective
The primary objective of this notebook is to establish baseline performance metrics for all datasets. These baselines are essential for comparison with advanced models in subsequent stages such as NEO-Forecast and optimization frameworks.

### Methodology
We begin by mounting Google Drive and verifying the existence of the dataset directory. The notebook then scans the base directory and recursively identifies all dataset folders, even if they are nested inside another directory.

For each dataset:
- The main data file (CSV or Parquet) is automatically detected
- A suitable target variable is inferred
- Time-based features and lag features are generated when applicable
- Baseline models are trained and evaluated

### Models Used
- Linear Regression
- Random Forest
- XGBoost (if available)
- MLP (Neural Network baseline)

### Output
The notebook generates:
- Dataset-wise baseline metrics (RMSE, MAE, R²)
- Consolidated results file
- Best model per dataset
- Execution log

All outputs are stored in:

MyDrive/PhD_Thesis_DataPipeline/baseline_results/

### Importance
This stage establishes a scientifically valid benchmark for evaluating advanced models. These results will be directly used in thesis chapters, comparison tables, and IEEE publications.

### Next Step
The next notebook in the pipeline will be:

03_NEO_Forecast_v1.ipynb

which will implement advanced forecasting architectures.

In [1]:
# ============================================================
# 02_baseline_models.ipynb
# BUNDLE-BASED FINAL CODE CELL
# ============================================================

import os
import gc
import json
import shutil
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
BUNDLES_DIR = Path("/content/drive/MyDrive/Processed Datasets Bundles")
OUTPUT_DIR = Path("/content/drive/MyDrive/baseline_results_from_bundles")
EXTRACT_DIR = OUTPUT_DIR / "extracted_bundles"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] BUNDLES_DIR: {BUNDLES_DIR}")
print(f"[INFO] OUTPUT_DIR:  {OUTPUT_DIR}")
print(f"[INFO] EXTRACT_DIR: {EXTRACT_DIR}")

if not BUNDLES_DIR.exists():
    raise FileNotFoundError(f"BUNDLES_DIR not found: {BUNDLES_DIR}")

print("\n[DEBUG] Top-level contents of BUNDLES_DIR:")
for p in sorted(BUNDLES_DIR.iterdir()):
    print(" -", p.name, "|", "DIR" if p.is_dir() else "FILE")

# ------------------------------------------------------------
# 2. Find all ZIP bundles
# ------------------------------------------------------------
zip_bundles = sorted(BUNDLES_DIR.rglob("*.zip"))

print(f"\n[INFO] ZIP bundles found: {len(zip_bundles)}")
for z in zip_bundles[:50]:
    print(" -", z.name)

if len(zip_bundles) == 0:
    raise ValueError("No ZIP bundles found inside 'Processed Datasets Bundles'.")

# ------------------------------------------------------------
# 3. Extract all bundles
# ------------------------------------------------------------
for i, zf in enumerate(zip_bundles, start=1):
    target = EXTRACT_DIR / f"bundle_{i:02d}_{zf.stem}"
    target.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(target)
        print(f"[EXTRACTED] {zf.name} -> {target.name}")
    except Exception as e:
        print(f"[WARN] Failed to extract {zf.name}: {e}")

# ------------------------------------------------------------
# 4. Find usable extracted data files
# ------------------------------------------------------------
DATA_EXTS = [".csv", ".parquet", ".xlsx", ".xls"]
MAX_ROWS = 50000

data_files = []
for ext in DATA_EXTS:
    data_files.extend(EXTRACT_DIR.rglob(f"*{ext}"))

data_files = sorted(set(data_files))

print(f"\n[INFO] Extracted usable data files found: {len(data_files)}")
for f in data_files[:100]:
    print(" -", f)

if len(data_files) == 0:
    raise ValueError("No usable data files found after extracting ZIP bundles.")

# ------------------------------------------------------------
# 5. ML imports
# ------------------------------------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# 6. Helper functions
# ------------------------------------------------------------
def load_table_fast(file_path: Path):
    ext = file_path.suffix.lower()

    if ext == ".csv":
        return pd.read_csv(file_path, nrows=MAX_ROWS, low_memory=False)

    if ext == ".parquet":
        df = pd.read_parquet(file_path)
        return df.head(MAX_ROWS)

    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(file_path, nrows=MAX_ROWS)

    raise ValueError(f"Unsupported file type: {file_path}")

def choose_target(df: pd.DataFrame):
    preferred_keys = ["load", "power", "energy", "target", "demand", "output", "appliances"]
    for c in df.columns:
        cl = str(c).lower()
        if any(k in cl for k in preferred_keys):
            return c

    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    return num_cols[-1] if len(num_cols) > 0 else None

def evaluate(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

def build_model(name):
    if name == "LR":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", LinearRegression())
        ])

    if name == "RF":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", RandomForestRegressor(
                n_estimators=30,
                random_state=42,
                n_jobs=-1
            ))
        ])

    raise ValueError(f"Unknown model: {name}")

# ------------------------------------------------------------
# 7. Run fast baseline on each extracted data file
# ------------------------------------------------------------
results = []
log = []

for file in data_files:
    print(f"\n[PROCESSING] {file.name}")

    try:
        df = load_table_fast(file)

        if df is None or len(df) < 50:
            raise ValueError(f"Dataset too small after loading: {0 if df is None else len(df)} rows")

        # try numeric conversion on object columns
        df = df.copy()
        for c in df.columns:
            if df[c].dtype == "object":
                df[c] = pd.to_numeric(df[c], errors="ignore")

        target = choose_target(df)
        if target is None:
            raise ValueError("No target column found")

        df_num = df.select_dtypes(include=np.number).dropna()

        if target not in df_num.columns:
            raise ValueError("Chosen target is not numeric after cleaning")

        X = df_num.drop(columns=[target])
        y = df_num[target]

        if X.shape[1] == 0:
            raise ValueError("No numeric feature columns available")

        split = int(0.8 * len(df_num))
        if split <= 0 or split >= len(df_num):
            raise ValueError("Invalid train-test split")

        X_train, X_test = X.iloc[:split], X.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]

        for model_name in ["LR", "RF"]:
            model = build_model(model_name)
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            metrics = evaluate(y_test, preds)

            results.append({
                "bundle_file": file.parent.parent.name if file.parent.parent.exists() else "",
                "data_file": file.name,
                "model": model_name,
                "n_rows": len(df_num),
                "n_features": X.shape[1],
                **metrics
            })

        log.append({
            "data_file": str(file),
            "status": "success"
        })

    except Exception as e:
        log.append({
            "data_file": str(file),
            "status": "failed",
            "error": str(e)
        })

    gc.collect()

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------
results_df = pd.DataFrame(results)
log_df = pd.DataFrame(log)

results_df.to_csv(OUTPUT_DIR / "baseline_results.csv", index=False)
log_df.to_csv(OUTPUT_DIR / "execution_log.csv", index=False)

summary = {
    "bundles_dir": str(BUNDLES_DIR),
    "zip_bundles_found": len(zip_bundles),
    "usable_data_files_found": len(data_files),
    "max_rows_per_file": MAX_ROWS,
    "models_run": ["LR", "RF"],
    "successful_files": int((log_df["status"] == "success").sum()) if len(log_df) else 0,
    "failed_files": int((log_df["status"] == "failed").sum()) if len(log_df) else 0,
}
with open(OUTPUT_DIR / "baseline_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# ------------------------------------------------------------
# 9. Create downloadable ZIP of baseline outputs
# ------------------------------------------------------------
zip_base = "/content/baseline_results_from_bundles"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("\n[SUCCESS] Bundle-based baseline run completed.")
print(f"[INFO] Results saved in: {OUTPUT_DIR}")
print(f"[INFO] Download ZIP ready: {zip_base}.zip")

display(results_df)
display(log_df)

from google.colab import files
files.download(f"{zip_base}.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] BUNDLES_DIR: /content/drive/MyDrive/Processed Datasets Bundles
[INFO] OUTPUT_DIR:  /content/drive/MyDrive/baseline_results_from_bundles
[INFO] EXTRACT_DIR: /content/drive/MyDrive/baseline_results_from_bundles/extracted_bundles

[DEBUG] Top-level contents of BUNDLES_DIR:
 - 01_data_preprocessing.ipynb | FILE
 - 01_full_preprocessing_bundle.zip | FILE
 - 01_manifests_and_reports.zip | FILE
 - 01_processed_datasets.zip | FILE
 - PROCESSED DATA BUNDLES.docx | FILE

[INFO] ZIP bundles found: 3
 - 01_full_preprocessing_bundle.zip
 - 01_manifests_and_reports.zip
 - 01_processed_datasets.zip
[EXTRACTED] 01_full_preprocessing_bundle.zip -> bundle_01_01_full_preprocessing_bundle
[EXTRACTED] 01_manifests_and_reports.zip -> bundle_02_01_manifests_and_reports
[EXTRACTED] 01_processed_datasets.zip -> bundle_03_01_processed_datasets

[INFO] Extracted usable data file

,bundle_file,data_file,model,n_rows,n_features,RMSE,MAE,R2
0,processed,uci_household_power_clean_hourly.parquet,LR,33418,21,2.027372e-02,1.326750e-02,0.999274
1,processed,uci_household_power_clean_hourly.parquet,RF,33418,21,2.163159e-02,1.342550e-02,0.999174
2,processed,uci_household_power_clean_raw.parquet,LR,49814,21,6.132763e-02,3.822605e-02,0.997888
3,processed,uci_household_power_clean_raw.parquet,RF,49814,21,4.052087e-02,2.002535e-02,0.999078
4,processed,electricity_load_diagrams_clean_raw.parquet,LR,50000,47,0.000000e+00,0.000000e+00,1.000000
5,processed,electricity_load_diagrams_clean_raw.parquet,RF,50000,47,0.000000e+00,0.000000e+00,1.000000
6,processed,edges.csv,LR,186,12,0.000000e+00,0.000000e+00,1.000000
7,processed,edges.csv,RF,186,12,0.000000e+00,0.000000e+00,1.000000
8,processed,generators.csv,LR,54,20,0.000000e+00,0.000000e+00,1.000000
9,processed,generators.csv,RF,54,20,0.000000e+00,0.000000e+00,1.000000


,data_file,status,error
0,/content/drive/MyDrive/baseline_results_from_b...,failed,Dataset too small after loading: 12 rows
1,/content/drive/MyDrive/baseline_results_from_b...,success,NaN
2,/content/drive/MyDrive/baseline_results_from_b...,success,NaN
3,/content/drive/MyDrive/baseline_results_from_b...,success,NaN
4,/content/drive/MyDrive/baseline_results_from_b...,failed,Invalid train-test split
5,/content/drive/MyDrive/baseline_results_from_b...,failed,Dataset too small after loading: 2 rows
6,/content/drive/MyDrive/baseline_results_from_b...,failed,Dataset too small after loading: 49 rows
7,/content/drive/MyDrive/baseline_results_from_b...,failed,Dataset too small after loading: 1 rows
8,/content/drive/MyDrive/baseline_results_from_b...,failed,Dataset too small after loading: 49 rows
9,/content/drive/MyDrive/baseline_results_from_b...,success,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>